# Day 1: Advanced Python Functions & Strict Type Hinting (Pydantic)

Welcome to Day 1 of your AI Engineering journey!

## Core Theory (Just-in-Time)

**Why Type Hinting and Pydantic?**
In traditional software engineering, inputs are often predictable (e.g., from a controlled frontend or internal API). In AI Engineering, however, your most critical component—the Large Language Model (LLM)—outputs probabilistic, unstructured, and often unpredictable text. 

To build reliable AI pipelines, you must enforce structure at the boundaries. 

**Pydantic** is the industry standard for this. It goes beyond static type checking (like `mypy`); it performs runtime data validation. If an LLM is supposed to return a JSON object containing a `score` (int) and a `summary` (string), Pydantic ensures that output actually conforms to that schema, throwing a clear error if the LLM hallucinated a different format. 

**Advanced Functions (Decorators & *args/**kwargs)**
AI workflows often require cross-cutting concerns like logging execution time (LLM calls are slow!), retrying failed calls (network hiccups or rate limits), or caching. Decorators are the clean, Pythonic way to wrap function execution without cluttering the business logic.

**AI Security Implications (PII, Prompt Injection, Fallbacks)**
Security is paramount in AI Engineering. When interacting with LLMs, three main areas must be addressed:
1. **PII Protection**: Never send Personally Identifiable Information (PII) to an external LLM. Use Pydantic or custom validators to redact sensitive information before the API call.
2. **Prompt Injection**: Malicious users may attempt to bypass system instructions. Robust input validation and strict output schemas (like those enforced by Pydantic) help mitigate the risk of successful prompt injections.
3. **Fallback Mechanisms**: LLMs can fail, hallucinate, or become unavailable. Always implement safe fallback strategies (e.g., default responses, retries) to ensure the system degrades gracefully.

## Code Implementation

Let's walk through a tiered progression of how to apply strict type hinting and decorators in your code, moving from basic concepts to production-ready patterns.

### Basic: Isolating the Core Concept
Here is a simple Pydantic model enforcing type validation and simulating basic fallback mechanisms with minimal boilerplate.

In [1]:
from pydantic import BaseModel, Field, ValidationError

class SimpleResponse(BaseModel):
    score: int
    summary: str

def process_llm_response(data: dict) -> SimpleResponse:
    try:
        return SimpleResponse(**data)
    except ValidationError as e:
        print(f"Validation error:\n{e}\nUsing fallback.")
        # Fallback mechanism
        return SimpleResponse(score=0, summary="Fallback summary")

# Validating correct input
valid_data = {"score": 95, "summary": "Great result"}
parsed_valid = process_llm_response(valid_data)
print("Basic Valid:", parsed_valid)

# Validating incorrect input triggering fallback
invalid_data = {"score": "ninety-five", "summary": "Bad result"}
parsed_invalid = process_llm_response(invalid_data)
print("\nBasic Fallback Applied:", parsed_invalid)

Basic Valid: score=95 summary='Great result'
Validation error:
1 validation error for SimpleResponse
score
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='ninety-five', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing
Using fallback.

Basic Fallback Applied: score=0 summary='Fallback summary'


### Medium: Interacting Concepts
Now, let's emphasize clean Object-Oriented Programming (OOP) and state management. We'll build a simple `LLMProcessor` class that uses a timing decorator and Pydantic validation.

In [2]:
import time
from typing import Callable, Any, Dict
from pydantic import BaseModel, ValidationError

class AnalysisResult(BaseModel):
    sentiment: str
    confidence: float

def simple_timer(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start = time.time()
        result = func(*args, **kwargs)
        print(f"[{func.__name__}] took {time.time() - start:.4f}s")
        return result
    return wrapper

class LLMProcessor:
    def __init__(self, fallback_sentiment: str = "neutral"):
        self.fallback_sentiment = fallback_sentiment
        self.processed_count = 0  # State management

    @simple_timer
    def process(self, raw_output: Dict[str, Any]) -> AnalysisResult:
        self.processed_count += 1
        try:
            # Simulate work
            time.sleep(0.05)
            return AnalysisResult(**raw_output)
        except ValidationError:
            print("Validation failed, utilizing fallback state.")
            return AnalysisResult(sentiment=self.fallback_sentiment, confidence=0.0)

processor = LLMProcessor()
result_good = processor.process({"sentiment": "positive", "confidence": 0.9})
print(f"Good Result: {result_good}")

result_bad = processor.process({"sentiment": "unknown", "confidence": "high"})
print(f"Bad Result (Fallback): {result_bad}")
print(f"Total processed calls: {processor.processed_count}")

[process] took 0.0503s
Good Result: sentiment='positive' confidence=0.9


Validation failed, utilizing fallback state.
[process] took 0.0505s
Bad Result (Fallback): sentiment='neutral' confidence=0.0
Total processed calls: 2


### Advanced: Production-Grade Implementation
This example provides a full production-ready implementation prioritizing clean OOP, strict type hinting, docstrings, exact import syntax, and AI Security (PII Redaction and Fallbacks). We will define a `SecureLLMClient` that redacts emails before processing.

In [3]:
import time
import logging
import re
from typing import Callable, Any, Dict, List, Optional
from functools import wraps
from pydantic import BaseModel, Field, ValidationError

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def log_execution_time(func: Callable[..., Any]) -> Callable[..., Any]:
    """
    A decorator that logs the execution time of a function.
    Crucial for monitoring LLM response times in production.
    """
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.time()
        logger.info(f"Executing {func.__name__}...")
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error(f"Error in {func.__name__}: {str(e)}")
            raise e
        end_time = time.time()
        logger.info(f"Finished {func.__name__} in {end_time - start_time:.4f} seconds")
        return result
    return wrapper

# --- Pydantic Models for Strict Validation ---

class UserInsight(BaseModel):
    """
    Defines the exact structure we expect from our LLM.
    Using Field allows us to add constraints and descriptions.
    """
    user_intent: str = Field(..., description="The detected intent of the user")
    risk_score: float = Field(..., ge=0.0, le=1.0, description="Risk score between 0.0 and 1.0")
    extracted_entities: List[str] = Field(default_factory=list, description="List of named entities found in text")

class SecureLLMClient:
    """
    A production-ready client simulating LLM interactions with built-in PII redaction
    and fallback mechanisms via rigorous OOP.
    """
    EMAIL_REGEX = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'

    def __init__(self, fallback_intent: str = "unknown"):
        self.fallback_intent = fallback_intent

    def redact_pii(self, text: str) -> str:
        """
        Redacts emails from text to ensure PII is not sent to the LLM.
        """
        redacted = re.sub(self.EMAIL_REGEX, "[REDACTED_EMAIL]", text)
        if redacted != text:
            logger.warning("PII detected and redacted from input.")
        return redacted

    @log_execution_time
    def process_prompt(self, prompt: str, mock_llm_response: Dict[str, Any]) -> UserInsight:
        """
        Simulates sending a prompt to an LLM and strictly validating the response.
        """
        safe_prompt = self.redact_pii(prompt)
        logger.info(f"Sending safe prompt: '{safe_prompt}'")
        
        try:
            # Simulate parsing JSON response from LLM
            validated_data = UserInsight(**mock_llm_response)
            return validated_data
        except ValidationError as e:
            logger.error(f"LLM output failed validation: {e}")
            logger.info("Utilizing safe fallback mechanism.")
            # Fallback mechanism
            return UserInsight(
                user_intent=self.fallback_intent,
                risk_score=1.0,  # Assume high risk on failure
                extracted_entities=[]
            )

# --- Example Usage ---
if __name__ == "__main__":
    client = SecureLLMClient()

    print("\n--- Testing Valid Output & PII Redaction ---")
    good_output = {
        "user_intent": "purchase",
        "risk_score": 0.1,
        "extracted_entities": ["laptop", "mouse"]
    }
    insight = client.process_prompt("I want to buy a laptop. My email is user@example.com", good_output)
    print(f"Parsed object: {insight}")

    print("\n--- Testing Invalid Output & Fallback ---")
    bad_output = {
        "risk_score": 5.0, # > 1.0, will cause validation error
        "extracted_entities": ["Error"]
        # Missing user_intent
    }
    fallback_insight = client.process_prompt("Check my account status.", bad_output)
    print(f"Fallback object: {fallback_insight}")


2026-08-20 12:49:54,309 - INFO - Executing process_prompt...


2026-08-20 12:49:54,311 - WARNING - PII detected and redacted from input.


2026-08-20 12:49:54,312 - INFO - Sending safe prompt: 'I want to buy a laptop. My email is [REDACTED_EMAIL]'


2026-08-20 12:49:54,313 - INFO - Finished process_prompt in 0.0041 seconds


2026-08-20 12:49:54,314 - INFO - Executing process_prompt...


2026-08-20 12:49:54,315 - INFO - Sending safe prompt: 'Check my account status.'


2026-08-20 12:49:54,316 - ERROR - LLM output failed validation: 2 validation errors for UserInsight
user_intent
  Field required [type=missing, input_value={'risk_score': 5.0, 'extr...ed_entities': ['Error']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
risk_score
  Input should be less than or equal to 1 [type=less_than_equal, input_value=5.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


2026-08-20 12:49:54,317 - INFO - Utilizing safe fallback mechanism.


2026-08-20 12:49:54,318 - INFO - Finished process_prompt in 0.0037 seconds



--- Testing Valid Output & PII Redaction ---
Parsed object: user_intent='purchase' risk_score=0.1 extracted_entities=['laptop', 'mouse']

--- Testing Invalid Output & Fallback ---
Fallback object: user_intent='unknown' risk_score=1.0 extracted_entities=[]


## Common Pitfalls in Production

1.  **Trusting LLM JSON Natively:** Never use raw `json.loads(llm_response)` and just assume the keys you need are present. LLMs hallucinate keys, change types (e.g., sending the string `"true"` instead of the boolean `True`), or return lists instead of dicts. Always pass the raw dict into a Pydantic model immediately.
2.  **Failing to Redact PII Early:** Accidentally leaking user data (like emails, SSNs, or API keys) in prompts is a major AI security vulnerability. Implement redaction steps *before* the prompt hits the network.
3.  **Missing Fallback Mechanisms:** If an LLM is down or outputs entirely un-parseable text, your pipeline should not crash. Ensure you have default behaviors (e.g., fallback responses, graceful degradation) to handle LLM failures.
4.  **Overuse of `Any` or `dict` Type Hints:** Using `def process(data: dict) -> dict:` defeats the purpose of type hinting. If you don't know the exact schema, use Pydantic's `model_validator` or define a flexible schema, but try to avoid raw dicts moving through your core business logic.

## Reference Links

- [Pydantic Official Documentation](https://docs.pydantic.dev/latest/)
- [Python Type Hinting (PEP 484)](https://peps.python.org/pep-0484/)
- [Python Decorators (Real Python)](https://realpython.com/primer-on-python-decorators/)

## Practical Lab / Homework

**Task:** Refactor a legacy document processing script using clean OOP principles and AI security best practices.

Below is a messy, legacy Python script that simulates fetching document metadata and a summary from an "API" (which in the future will be an LLM). 

**Your Requirements:**
1.  **Clean OOP:** Wrap the logic in a class named `DocumentProcessor`.
2.  **Strict Typing:** Replace the raw dictionary return type with a Pydantic `BaseModel` called `DocumentSummary`.
    *   It should have a `doc_id` (string), `title` (string), `tags` (list of strings), and `word_count` (integer greater than 0).
3.  **Decorator:** Write a retry decorator called `@retry_on_failure` that will retry the function up to 3 times if an Exception is thrown, with a 1-second delay between attempts.
4.  **AI Security (Fallback):** In case of ultimate failure, implement a fallback mechanism within `DocumentProcessor` that returns a safe default `DocumentSummary` rather than crashing.
5.  **Video Walkthrough:** Record a brief async video (2-3 minutes) explaining your design decisions, specifically focusing on how you integrated OOP, state, and security considerations. (Note: You do not actually need to upload the video here, but treat this as a mandatory step in your local workflow!).

In [4]:
# --- LEGACY CODE (Refactor this!) ---

import random
import time

def fetch_document_summary(doc_id):
    """Simulates a flaky API call that returns unstructured data."""
    # Simulate flakiness
    if random.random() < 0.5:
        raise ConnectionError("API Connection reset!")
        
    # Returns a messy dict
    return {
        "id": doc_id,
        "title": "Understanding Attention Mechanisms",
        "tags": ["AI", "Transformers", "NLP"],
        "words": 1500
    }


# --- YOUR REFACTORED CODE BELOW ---
from pydantic import BaseModel, Field, ValidationError
from typing import Callable, Any, Dict
from functools import wraps
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

class DocumentSummary(BaseModel):
    doc_id: str
    title: str
    tags: list[str] = Field(default_factory=list)
    word_count: int = Field(..., gt=0)

def retry_on_failure(max_retries: int = 3, delay: float = 1.0) -> Callable[..., Any]:
    """
    Decorator to retry a function if it raises an Exception.
    """
    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            retries = 0
            while retries < max_retries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    retries += 1
                    logger.warning(f"Attempt {retries} failed: {e}. Retrying in {delay} seconds...")
                    time.sleep(delay)
            raise Exception(f"Failed after {max_retries} attempts")
        return wrapper
    return decorator

class DocumentProcessor:
    """
    Production-grade document processor utilizing strict typing, retries,
    and AI Security fallback mechanisms.
    """
    def __init__(self, fallback_title: str = "Unknown Document"):
        self.fallback_title = fallback_title
        self.processed_count = 0

    @retry_on_failure(max_retries=3, delay=0.5)
    def _fetch_raw(self, doc_id: str) -> Dict[str, Any]:
        # This wraps the flaky legacy call
        return fetch_document_summary(doc_id)

    def process(self, doc_id: str) -> DocumentSummary:
        self.processed_count += 1
        try:
            raw_data = self._fetch_raw(doc_id)
            
            # Map legacy dictionary keys to Pydantic model
            mapped_data = {
                "doc_id": raw_data.get("id", ""),
                "title": raw_data.get("title", ""),
                "tags": raw_data.get("tags", []),
                "word_count": raw_data.get("words", 0)
            }
            return DocumentSummary(**mapped_data)
        except Exception as e:
            logger.error(f"Final failure fetching document {doc_id}: {e}")
            logger.info("Utilizing safe fallback summary.")
            return DocumentSummary(
                doc_id=doc_id,
                title=self.fallback_title,
                tags=["error", "fallback"],
                word_count=1
            )

# --- Test your refactored code ---
if __name__ == "__main__":
    processor = DocumentProcessor()
    
    summary = processor.process("doc-123")
    print(f"\nResult: {summary}")



Result: doc_id='doc-123' title='Understanding Attention Mechanisms' tags=['AI', 'Transformers', 'NLP'] word_count=1500
